In [ ]:
import pandas as pd 
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'../..')
from models import MetaEvaluator
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp

In [ ]:
FINAL_FEATURE_FRACTION = {'powersupply': 50, 'airlines': 65, 'electricity': 15, 'rialto': 10}
elec_eval = MetaEvaluator(dataset_name="electricity", dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["electricity"]).fit()
powersupply_eval = MetaEvaluator(dataset_name="powersupply",dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["powersupply"]).fit()
airlines_eval = MetaEvaluator(dataset_name="airlines",dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["airlines"]).fit()
rialto_eval = MetaEvaluator(dataset_name="rialto",dir="henrique_st", feature_fraction=FINAL_FEATURE_FRACTION["rialto"]).fit()

In [ ]:

mean_results = [
    {
        "dataset": evaluator.dataset_name,
        "model": model,
        "metric": metric,
        "baseline": evaluator.results[model][f"{metric}_mse_baseline"].mean(),
        "original_mtl": evaluator.results[model][f"{metric}_original_mtl_mse"].mean(),
        "proposed_mtl": evaluator.results[model][f"{metric}_proposed_mtl_mse"].mean(),
    }
    for evaluator in (elec_eval, powersupply_eval, airlines_eval, rialto_eval)
    for model in evaluator.metrics.keys()
    for metric in ["f1-score"]
]
mean_results_df = pd.DataFrame(mean_results)
print(mean_results_df.head())
print(mean_results_df.shape)

aux = mean_results_df

In [ ]:
mean_results_df = mean_results_df[["baseline","original_mtl","proposed_mtl"]]
print(mean_results_df.shape)

In [ ]:

transposed_results = mean_results_df.T

baseline = transposed_results.loc['baseline'].values
original_mtl = transposed_results.loc['original_mtl'].values
proposed_mtl = transposed_results.loc['proposed_mtl'].values

friedman_statistics, friedman_p_value= friedmanchisquare(baseline,original_mtl,proposed_mtl)
print(f"Friedman Statistics: {friedman_statistics}")
print(f"Friedman p-value: {friedman_p_value}")

In [ ]:
nemenyi_results = sp.posthoc_nemenyi_friedman(mean_results_df)

In [ ]:
print(nemenyi_results)
alpha = 0.05
print(nemenyi_results<alpha)

In [ ]:
mean_results_df

In [ ]:
ordered_indexes =  mean_results_df.rank(axis=1,ascending=True).astype(int)
print(ordered_indexes)

In [ ]:
baseline_better = (ordered_indexes["baseline"]==1)
baseline_better_indexes = ordered_indexes[baseline_better].index
print(f"baseline_better: {baseline_better_indexes.shape[0]}")
print(aux.loc[baseline_better_indexes][["dataset","model","metric"]])

original_mtl_better = (ordered_indexes["original_mtl"]==1)
original_mtl_better_indexes = (ordered_indexes[original_mtl_better]==1).index
print(f"original_mtl_better: {original_mtl_better_indexes.shape[0]}")
print(aux.loc[original_mtl_better_indexes][["dataset","model","metric"]])

proposed_mtl_better = (ordered_indexes["proposed_mtl"]==1)
proposed_mtl_better_indexes = (ordered_indexes[proposed_mtl_better]==1).index
print(f"proposed_mtl_better: { proposed_mtl_better_indexes.shape[0]}" )